<font face="Comic Sans MS">Импорт необходимых библиотек:</font>

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from tqdm import tqdm

Создание модели для получение вектора на 512 признаков 

In [ ]:
def get_model_resnet50_embedding():
    """
    Функция возвращает модель resnet50\n
    с модификацией для создания embedding
    """

    model = models.resnet50(weights='IMAGENET1K_V1')
    model.fc = nn.Linear(2048, 512)

    return model

Создание Датасета и Лоадера

In [ ]:
transform = transforms.Compose([
        transforms.Resize((1280, 763)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

dataset = datasets.ImageFolder('noisy_dataset', transform=transform)

train_loader = DataLoader(dataset, batch_size=5, shuffle=True)

Обучение

In [ ]:
def train_parking_zone_detect(model, train_loader, epochs=15, lr=5e-5):

    device = 'cuda:0'
    model.to(device)

    parking_names = train_loader.dataset.classes
    num_parking = len(parking_names)
    print(f'Найдено парковок: {num_parking} -> {parking_names}')


    loss_func = ArcFaceLoss(
        num_classes=num_parking, 
        embedding_size=512,
        margin=28.6,
        scale=64
    ).to(device)

    optimizer = AdamW([
        {'params': model.parameters()},
        {'params': loss_func.parameters()}
    ], lr=lr)

    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    pbar = tqdm(range(epochs))

    best_loss = float('inf')

    model.train()
    for epoch in pbar:

        losses = []

        for images, labels in enumerate(train_loader):  # Зачем enumerate?

            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            embeddings = model(images)

            loss = loss_func(embeddings, labels)
            loss.backward()
            losses.append(loss.item())

            optimizer.step()

        scheduler.step()

        epoch_loss = torch.Tensor(losses).mean().item()

        if epoch_loss < best_loss:
            best_loss = epoch_loss

            torch.save(model.state_dict(), 'best_parking_space_backbone.pth')
            torch.save(loss_func.state_dict(), 'best_parking_space_centers.pth')

        pbar.set_postfix({'loss': epoch_loss, 'best_loss': best_loss})

    pbar.close()
    torch.save(model.state_dict(), 'last_parking_space_backbone.pth')
    torch.save(loss_func.state_dict(), 'last_parking_space_centers.pth')

    with open('parking_names.txt', 'w') as f:
        f.write("\n".join(parking_names))

In [ ]:
model_resnet50 = get_model_resnet50_embedding()
train_parking_zone_detect(model_resnet50, train_loader, epochs=30)